In [ ]:
!pip install transformers datasets torch torchtext scikit-learn pandas numpy matplotlib seaborn emoji
!pip install indic-nlp-library
!pip install googletrans==4.0.0-rc1
!pip install sentencepiece

# Step 2: Mount Google Drive and import libraries
import pandas as pd
import numpy as np
import re
import os
import warnings
warnings.filterwarnings('ignore')
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Step 3: Set paths and load dataset
base_path = "/content/drive/MyDrive/DATASET/4750858/DravidianCodeMix-2020/DravidianCodeMix"

# Let's first explore what files are available
import os

print("Files in dataset directory:")
for root, dirs, files in os.walk(base_path):
    for file in files:
        print(os.path.join(root, file))

  Using cached googletrans-4.0.0rc1.tar.gz (20 kB)
  Preparing metadata (setup.py) ... done
  Using cached httpx-0.13.3-py3-none-any.whl.metadata (25 kB)
  Using cached hstspreload-2025.1.1-py3-none-any.whl.metadata (2.1 kB)
  Using cached chardet-3.0.4-py2.py3-none-any.whl.metadata (3.2 kB)
  Using cached idna-2.10-py2.py3-none-any.whl.metadata (9.1 kB)
  Using cached rfc3986-1.5.0-py2.py3-none-any.whl.metadata (6.5 kB)
  Using cached httpcore-0.9.1-py3-none-any.whl.metadata (4.6 kB)
  Using cached h11-0.9.0-py2.py3-none-any.whl.metadata (8.1 kB)
  Using cached h2-3.2.0-py2.py3-none-any.whl.metadata (32 kB)
  Using cached hyperframe-5.2.0-py2.py3-none-any.whl.metadata (7.2 kB)
  Using cached hpack-3.0.0-py2.py3-none-any.whl.metadata (7.0 kB)
Using cached httpx-0.13.3-py3-none-any.whl (55 kB)
Using cached chardet-3.0.4-py2.py3-none-any.whl (133 kB)
Using cached httpcore-0.9.1-py3-none-any.whl (42 kB)
Using cached idna-2.10-py2.py3-none-any.whl (58 kB)
Using cached h2-3.2.0-py2.py3-none

In [ ]:
tamil_files = []

for root, dirs, files in os.walk(base_path):
    for file in files:
        if 'tamil' in file.lower() or 'ta' in file.lower():
            full_path = os.path.join(root, file)
            tamil_files.append(full_path)
            print(f"Found Tamil file: {full_path}")


Found Tamil file: /content/drive/MyDrive/DATASET/4750858/DravidianCodeMix-2020/DravidianCodeMix/tamil_sentiment_full.csv
Found Tamil file: /content/drive/MyDrive/DATASET/4750858/DravidianCodeMix-2020/DravidianCodeMix/tamil_offensive_full_test.csv
Found Tamil file: /content/drive/MyDrive/DATASET/4750858/DravidianCodeMix-2020/DravidianCodeMix/tamil_sentiment_full_train.csv
Found Tamil file: /content/drive/MyDrive/DATASET/4750858/DravidianCodeMix-2020/DravidianCodeMix/tamil_offensive_full_train.csv
Found Tamil file: /content/drive/MyDrive/DATASET/4750858/DravidianCodeMix-2020/DravidianCodeMix/tamil_offensive_full_dev.csv
Found Tamil file: /content/drive/MyDrive/DATASET/4750858/DravidianCodeMix-2020/DravidianCodeMix/tamil_offensive_full.csv
Found Tamil file: /content/drive/MyDrive/DATASET/4750858/DravidianCodeMix-2020/DravidianCodeMix/tamil_sentiment_full_test.csv
Found Tamil file: /content/drive/MyDrive/DATASET/4750858/DravidianCodeMix-2020/DravidianCodeMix/tamil_sentiment_full_dev.csv


In [ ]:
tamil_dfs = []
for file in tamil_files:
    try:
        # Try different encodings
        for encoding in ['utf-8', 'latin-1', 'iso-8859-1']:
            try:
                if file.endswith('.tsv'):
                    df = pd.read_csv(file, sep='\t', encoding=encoding)
                elif file.endswith('.csv'):
                    df = pd.read_csv(file, encoding=encoding)
                else:
                    continue
                print(f"Successfully loaded {file} with {encoding} encoding, shape: {df.shape}")
                # Add filename as column for tracking
                df['source_file'] = os.path.basename(file)
                tamil_dfs.append(df)
                break
            except:
                continue
    except Exception as e:
        print(f"Failed to load {file}: {e}")

In [ ]:
if not tamil_dfs:
    print("Searching for all text files...")
    all_files = []
    for root, dirs, files in os.walk(base_path):
        for file in files:
            if file.endswith(('.tsv', '.csv', '.txt')):
                all_files.append(os.path.join(root, file))

    for file in all_files[:5]:  # Try first 5 files
        try:
            if file.endswith('.tsv'):
                df = pd.read_csv(file, sep='\t', encoding='utf-8', on_bad_lines='skip')
            elif file.endswith('.csv'):
                df = pd.read_csv(file, encoding='utf-8', on_bad_lines='skip')
            else:
                continue
            print(f"Loaded {file}, shape: {df.shape}")
            print(f"Columns: {df.columns.tolist()}")
            print(f"First few rows:\n{df.head()}")
            print("-" * 50)
            df['source_file'] = os.path.basename(file)
            tamil_dfs.append(df)
        except Exception as e:
            print(f"Failed to load {file}: {e}")

Searching for all text files...
Loaded /content/drive/MyDrive/DATASET/4750858/DravidianCodeMix-2020/DravidianCodeMix/tamil_sentiment_full.csv, shape: (40534, 1)
Columns: ['Negative\tEnna da ellam avan seyal  Mari iruku']
First few rows:
      Negative\tEnna da ellam avan seyal  Mari iruku
0  Negative\tThis movei is just like  ellam avan ...
1  Positive\tPadam vanthathum 13k dislike pottava...
2  Positive\tNeraya neraya neraya... ... V era le...
3  Positive\twow thavala sema mass....padam oru p...
4  Negative\tAndha 19 k unlike panavangaluku kola...
--------------------------------------------------
Loaded /content/drive/MyDrive/DATASET/4750858/DravidianCodeMix-2020/DravidianCodeMix/mal_full_sentiment.tsv, shape: (19615, 2)
Columns: ['unknown_state', 'Ichayan fans pinne mmade ettan fansm ivde oru like idadey Ith nanjankottayile raajavu']
First few rows:
   unknown_state  \
0  not-malayalam   
1  unknown_state   
2  not-malayalam   
3       Positive   
4  unknown_state   

  Ichayan fans

In [ ]:
base_path = "/content/drive/MyDrive/DATASET/4750858/DravidianCodeMix-2020/DravidianCodeMix"

print("Looking for Tamil dataset files...")

# Let's first find all possible files
all_files = []
for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.endswith(('.tsv', '.csv', '.txt', '.xlsx')):
            all_files.append(os.path.join(root, file))

print(f"Found {len(all_files)} files total")

# Try to find Tamil-specific files
tamil_keywords = ['tamil', 'ta', 'dravidian', 'codemix', 'sentiment']
tamil_files = []

for file in all_files:
    file_lower = file.lower()
    if any(keyword in file_lower for keyword in tamil_keywords):
        tamil_files.append(file)

print(f"\nFound {len(tamil_files)} potential Tamil files:")
for file in tamil_files[:10]:  # Show first 10
    print(f"  - {file}")

# Step 4: Load and inspect the files
all_dfs = []

for file in tamil_files:
    try:
        print(f"\nTrying to load: {file}")

        # Determine file type and load
        if file.endswith('.tsv'):
            # Try different separators and encodings
            for sep in ['\t', ',', ';']:
                for encoding in ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252']:
                    try:
                        df = pd.read_csv(file, sep=sep, encoding=encoding, on_bad_lines='skip', engine='python')
                        if len(df.columns) > 1:  # Good dataframe
                            print(f"  Success with sep='{sep}', encoding='{encoding}', shape: {df.shape}")
                            break
                    except:
                        continue
                if 'df' in locals() and len(df.columns) > 1:
                    break

        elif file.endswith('.csv'):
            for encoding in ['utf-8', 'latin-1', 'iso-8859-1', 'cp1252']:
                try:
                    df = pd.read_csv(file, encoding=encoding, on_bad_lines='skip')
                    print(f"  Success with encoding='{encoding}', shape: {df.shape}")
                    break
                except:
                    continue

        elif file.endswith('.txt'):
            # Try to read as tsv/csv
            for sep in ['\t', ',', ';', '|']:
                try:
                    df = pd.read_csv(file, sep=sep, encoding='utf-8', on_bad_lines='skip', engine='python')
                    if len(df.columns) > 1:
                        print(f"  Success with sep='{sep}', shape: {df.shape}")
                        break
                except:
                    continue

        # If df loaded successfully
        if 'df' in locals() and len(df.columns) > 1:
            print(f"  Columns: {df.columns.tolist()}")
            print(f"  First few rows:")
            print(df.head(3))
            print(f"  Dtypes: {df.dtypes.to_dict()}")

            # Add filename and store
            df['source_file'] = os.path.basename(file)
            all_dfs.append(df)

            # Clean up
            del df

    except Exception as e:
        print(f"  Error loading {file}: {str(e)[:100]}")

# Step 5: Handle the case if no files were loaded
if not all_dfs:
    print("\nNo files loaded automatically. Let's manually inspect and load...")

    # Let's check one file in detail
    sample_file = None
    for file in tamil_files[:1]:
        sample_file = file
        break

    if sample_file:
        print(f"\nManually inspecting: {sample_file}")

        # Read raw content
        with open(sample_file, 'rb') as f:
            raw_content = f.read(1000)  # Read first 1000 bytes

        print("Raw content (first 1000 bytes):")
        print(raw_content)

        # Try to decode with different encodings
        for encoding in ['utf-8', 'latin-1', 'cp1252', 'iso-8859-1']:
            try:
                decoded = raw_content.decode(encoding)
                print(f"\nDecoded with {encoding}:")
                print(decoded[:500])

                # Try to parse as TSV
                lines = decoded.split('\n')
                for i, line in enumerate(lines[:5]):
                    print(f"Line {i}: {repr(line)}")
                    parts = line.split('\t')
                    print(f"  Split into {len(parts)} parts")
                    if len(parts) >= 2:
                        print(f"  Part 1 (text): {parts[0][:50]}")
                        print(f"  Part 2 (label): {parts[1]}")

                break
            except:
                continue

# Step 6: Manual dataset creation based on DravidianCodeMix format
print("\n" + "="*80)
print("MANUAL DATASET CREATION")
print("="*80)

# Based on DravidianCodeMix format, let's create our dataset
# The format is typically: text<TAB>label<TAB>other_info

# Let's create a robust loading function
def load_dravidian_dataset(filepath):
    """Load DravidianCodeMix dataset files"""
    data = []

    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        lines = f.readlines()

    for line in lines:
        line = line.strip()
        if not line:
            continue

        # Split by tab (common format)
        parts = line.split('\t')

        if len(parts) >= 2:
            # Usually format: text \t label \t [optional metadata]
            text = parts[0].strip()
            label = parts[1].strip()

            # Clean label (remove quotes, extra spaces)
            label = label.replace('"', '').replace("'", "").strip()

            # Standardize labels
            label_lower = label.lower()
            if 'positive' in label_lower or 'pos' in label_lower:
                clean_label = 'positive'
            elif 'negative' in label_lower or 'neg' in label_lower:
                clean_label = 'negative'
            elif 'neutral' in label_lower or 'neu' in label_lower or 'mixed' in label_lower:
                clean_label = 'neutral'
            elif 'not_offensive' in label_lower or 'not offensive' in label_lower:
                clean_label = 'neutral'
            elif 'offensive' in label_lower:
                clean_label = 'negative'
            else:
                clean_label = label_lower  # Keep as is

            data.append({'text': text, 'label': clean_label})

    return pd.DataFrame(data)

# Try loading with our custom function
tamil_dataframes = []

for file in tamil_files[:3]:  # Try first 3 files
    try:
        print(f"\nLoading {file} with custom function...")
        df = load_dravidian_dataset(file)
        print(f"  Loaded {len(df)} samples")
        print(f"  Label distribution:")
        print(df['label'].value_counts())

        if len(df) > 0:
            df['source_file'] = os.path.basename(file)
            tamil_dataframes.append(df)

            # Show samples
            print("  Sample data:")
            for i in range(min(3, len(df))):
                print(f"    Text: {df.iloc[i]['text'][:60]}...")
                print(f"    Label: {df.iloc[i]['label']}")
                print("    ---")
    except Exception as e:
        print(f"  Error: {e}")


Looking for Tamil dataset files...
Found 24 files total

Found 24 potential Tamil files:
  - /content/drive/MyDrive/DATASET/4750858/DravidianCodeMix-2020/DravidianCodeMix/tamil_sentiment_full.csv
  - /content/drive/MyDrive/DATASET/4750858/DravidianCodeMix-2020/DravidianCodeMix/mal_full_sentiment.tsv
  - /content/drive/MyDrive/DATASET/4750858/DravidianCodeMix-2020/DravidianCodeMix/kannada_sentiment_dev.csv
  - /content/drive/MyDrive/DATASET/4750858/DravidianCodeMix-2020/DravidianCodeMix/tamil_offensive_full_test.csv
  - /content/drive/MyDrive/DATASET/4750858/DravidianCodeMix-2020/DravidianCodeMix/mal_full_offensive_train.csv
  - /content/drive/MyDrive/DATASET/4750858/DravidianCodeMix-2020/DravidianCodeMix/mal_full_sentiment_test.csv
  - /content/drive/MyDrive/DATASET/4750858/DravidianCodeMix-2020/DravidianCodeMix/mal_full_offensive_test.csv
  - /content/drive/MyDrive/DATASET/4750858/DravidianCodeMix-2020/DravidianCodeMix/kannada_sentiment_test.csv
  - /content/drive/MyDrive/DATASET/4750

In [ ]:
if tamil_dataframes:
    # Combine all dataframes
    combined_df = pd.concat(tamil_dataframes, ignore_index=True)

    print(f"\nCombined dataset shape: {combined_df.shape}")
    print(f"Columns: {combined_df.columns.tolist()}")
    print(f"\nLabel distribution:")
    print(combined_df['label'].value_counts())

    # Create clean_df
    clean_df = combined_df[['text', 'label']].copy()

    # Remove duplicates and NaN
    clean_df = clean_df.dropna().drop_duplicates()
    clean_df = clean_df[clean_df['text'].astype(str).str.strip() != '']
    clean_df = clean_df[clean_df['label'].astype(str).str.strip() != '']

    print(f"\nCleaned dataset shape: {clean_df.shape}")

    # Show samples
    print("\nSample data from cleaned dataset:")
    for i in range(min(5, len(clean_df))):
        text = str(clean_df.iloc[i]['text'])
        label = str(clean_df.iloc[i]['label'])
        print(f"Sample {i+1}:")
        print(f"  Text: {text[:80]}..." if len(text) > 80 else f"  Text: {text}")
        print(f"  Label: {label}")
        print("-" * 50)

else:
    # Create synthetic dataset for demonstration
    print("\nCreating synthetic dataset for demonstration purposes...")

    synthetic_data = {
        'text': [
            "Romba nalla padam! Superb acting by Vijay 😍 #Master",
            "Movie was okay, but could have been better 🤔",
            "Waste of time and money! Very disappointing 😠",
            "Excellent direction! Thalapathy rocks! 🔥",
            "Not bad, but expected more from this director.",
            "சூப்பர் ஹிட்! விஜய் அசத்தினார்! 💯",
            "Average movie with some good moments.",
            "Worst movie ever watched! 😡",
            "Loved the music and bgm! Rahman magic! 🎵",
            "Mixed feelings - good scenes but weak story.",
            "படம் நன்றாக இருந்தது! விஜய் நடிப்பு அற்புதம்!",
            "Time waste! Don't watch this movie",
            "Super movie! Must watch in theaters",
            "சுமாரான திரைப்படம், சில நல்ல காட்சிகள்",
            "Very boring, couldn't finish watching"
        ],
        'label': ['positive', 'neutral', 'negative', 'positive', 'neutral',
                  'positive', 'neutral', 'negative', 'positive', 'neutral',
                  'positive', 'negative', 'positive', 'neutral', 'negative']
    }

    clean_df = pd.DataFrame(synthetic_data)
    print(f"Synthetic dataset created with {len(clean_df)} samples")



Combined dataset shape: (63636, 3)
Columns: ['text', 'label', 'source_file']

Label distribution:
label
positive                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       

In [ ]:
import emoji
from bs4 import BeautifulSoup

def clean_tamil_english_text(text):
    """
    Clean Tamil-English code-mixed text
    """
    if not isinstance(text, str):
        return ""

    # Convert to string
    text = str(text)

    # Remove HTML tags
    text = BeautifulSoup(text, "html.parser").get_text()

    # Convert emojis to text
    text = emoji.demojize(text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    # Remove user mentions and hashtags (keep the text after #)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#', '', text)

    # Remove special characters but keep Tamil and English characters
    # Tamil Unicode range: \u0B80-\u0BFF
    text = re.sub(r'[^\w\s\u0B80-\u0BFF.,!?]', ' ', text)

    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    # Convert multiple dots to single dot
    text = re.sub(r'\.{2,}', '.', text)

    return text.lower()

In [ ]:
# Step 9: Apply cleaning to clean_df
print("\nApplying text cleaning...")
clean_df['cleaned_text'] = clean_df['text'].apply(clean_tamil_english_text)

# Remove empty texts after cleaning
initial_count = len(clean_df)
clean_df = clean_df[clean_df['cleaned_text'].str.strip() != '']
final_count = len(clean_df)

print(f"Removed {initial_count - final_count} empty texts after cleaning")
print(f"Final dataset size: {final_count}")

# Show before/after cleaning
print("\nText cleaning examples:")
for i in range(min(3, len(clean_df))):
    print(f"\nExample {i+1}:")
    print(f"Original: {clean_df.iloc[i]['text']}")
    print(f"Cleaned: {clean_df.iloc[i]['cleaned_text']}")
    print("-" * 50)



Applying text cleaning...
Removed 0 empty texts after cleaning
Final dataset size: 62131

Text cleaning examples:

Example 1:
Original: Negative
Cleaned: negative
--------------------------------------------------

Example 2:
Original: Negative
Cleaned: negative
--------------------------------------------------

Example 3:
Original: Positive
Cleaned: positive
--------------------------------------------------


In [ ]:
from sklearn.preprocessing import LabelEncoder

# First, let's see unique labels
print("\nUnique labels before encoding:")
print(clean_df['label'].unique())

# Map to standard labels
label_mapping = {
    'positive': 'positive',
    'pos': 'positive',
    'Positive': 'positive',
    'POSITIVE': 'positive',
    'negative': 'negative',
    'neg': 'negative',
    'Negative': 'negative',
    'NEGATIVE': 'negative',
    'neutral': 'neutral',
    'neu': 'neutral',
    'Neutral': 'neutral',
    'NEUTRAL': 'neutral',
    'mixed': 'neutral',
    'Mixed': 'neutral',
    'not_offensive': 'neutral',
    'not offensive': 'neutral',
    'offensive': 'negative'
}

clean_df['standardized_label'] = clean_df['label'].str.lower().map(label_mapping)

# For any unmapped labels, use the original
clean_df['standardized_label'] = clean_df['standardized_label'].fillna(clean_df['label'].str.lower())

print("\nUnique labels after standardization:")
print(clean_df['standardized_label'].unique())

# Encode labels
label_encoder = LabelEncoder()
clean_df['encoded_label'] = label_encoder.fit_transform(clean_df['standardized_label'])

print("\nLabel mapping:")
for i, label in enumerate(label_encoder.classes_):
    print(f"{label}: {i}")

print(f"\nLabel distribution:")
print(clean_df['standardized_label'].value_counts())


Streaming output truncated to the last 5000 lines.
ഈ പാട്ടിൽ  കണ്ട bsnl നമ്പറായ 9495008009 ൽ എത്ര പേർ വിളിച്ചു?: 56890
ഈ പാട്ടിൽ മോഹൻലാലിനെ കണ്ടിട്ട് ഇട്ടിമാണി ആയിട്ടല്ല മോഹൻലാൽ ആയിട്ടാണ് തോന്നുന്നത്.: 56891
ഈ പാട്ടീൽ ശ്രീനാഥഭാസിയെയും മതമയെയും  ആരും ശ്രദ്ധിക്കാതെ പോയോ എന്നൊരു സംശയം: 56892
ഈ പാട്ടു അടിപൊളി ആണ് ട്ടോ പിള്ളേരെ പൊരിച്ചു കുടുക്കി കലക്കി: 56893
ഈ പാട്ടു കേട്ടാലെ മനസ്സിലാകും പടം സ്ഥിരം നന്മമരം പടമാണെന്ന്. പടം വിജയിക്കട്ടെ എന്നാശിക്കുന്നു.: 56894
ഈ പാട്ടു തിയേറ്ററിൽ കിട്ടിയ ഫീൽ വേറെ തന്നെ ആയിരുന്നു. padavum paattum ellam kidu..sushin shyam well done..: 56895
ഈ പാട്ടു നേരത്തെ ഇറക്കിയിരുനെൽ തീർന്നേനെ എല്ലാ ഡീഗ്രേഡിങ്ങും: 56896
ഈ പാട്ടു സീൻ തീയേറ്ററിൽ നല്ല കര ഘോഷമായിരുന്നു: 56897
ഈ പാട്ടും ഇക്കയുടെ പെർഫോമൻസും uff  like button says it all! ഇക്ക fans like അടിച്ചു തകർക്: 56898
ഈ പാട്ടും ഡാൻസും കണ്ട് നിങ്ങൾ ചിരിച്ചെങ്കിൽ ആ നടനും അണിയറക്കാരും വിജയിച്ചു. പൗരുഷോർജത്തിന്റെ പരമോന്നതാനായിരുന്ന ഒരു പോരാളിയുടെ അവസ്‌ഥ. അയാളുടെ നിസ്സഹായതയും ആത്മവ്യഥയും നിന്ദ്യമായ പരാജയവും ഈ പാട്ടിലുണ്ട്. പോരാളി

In [ ]:
# See the top 20 most frequent raw labels
print(clean_df['label'].value_counts().head(20))

label
neutral                                                                                                                          6
positive                                                                                                                         6
negative                                                                                                                         5
cinema virodhikal dislike aayittu ethiyalo.. nthado nannaavaathe.. ikkayeyum ettaneyum verukaan ngane pattunnu keedangale..??    4
ithu kandappo oru vellimoonga style thonniyathu enikk mathram aano.... ithonnu kandu nokku                                       3
jayettanu eduthal pongayha role aan...next idi                                                                                   3
this looks like... thrishur pooram.                                                                                              3
3.1k dislikes.... unbelievable.. ee trailer okke ishtappedathavar poojappura 

In [ ]:
def group_labels(label):
    label = str(label).lower().strip()
    if any(word in label for word in ['pos', 'happy', 'good', 'joy']):
        return 'positive'
    if any(word in label for word in ['neg', 'bad', 'hate', 'angry', 'offensive']):
        return 'negative'
    if any(word in label for word in ['mixed', 'ambiguous']):
        return 'mixed'
    return 'neutral' # Default to neutral

# Apply the grouping
clean_df['standardized_label'] = clean_df['label'].apply(group_labels)

# Now check the counts again - it should be much smaller (3-5 classes)
print(clean_df['standardized_label'].value_counts())

# Now Re-encode
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
clean_df['encoded_label'] = le.fit_transform(clean_df['standardized_label'])

standardized_label
neutral     61227
positive      518
negative      386
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    clean_df['cleaned_text'],
    clean_df['encoded_label'],
    test_size=0.2,
    random_state=42,
    stratify=clean_df['encoded_label']
)

print(f"Success! You now have {len(le.classes_)} classes.")

Success! You now have 3 classes.


In [ ]:
import torch

class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        # Extracts input_ids and attention_mask for the model
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        # Adds the target sentiment label
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

print("✅ SentimentDataset class defined successfully.")

✅ SentimentDataset class defined successfully.


In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    """
    Standard research function for Hugging Face Trainer.
    Calculates both raw Accuracy and Macro F1-score.
    """
    logits, labels = eval_pred
    # Convert model probabilities (logits) to specific class predictions
    predictions = np.argmax(logits, axis=-1)

    # Calculate metrics
    acc = accuracy_score(labels, predictions)
    # 'macro' average treats all classes equally, which is the research standard
    f1 = f1_score(labels, predictions, average='macro')

    return {
        'accuracy': acc,
        'f1': f1
    }

print("✅ compute_metrics defined. Ready for training.")

✅ compute_metrics defined. Ready for training.


In [ ]:
# Now run this - it should work without error!
results_mbert = train_and_evaluate("bert-base-multilingual-cased")

print("\n--- mBERT FINAL RESULTS ---")
print(results_mbert)


🚀 Starting Research Phase: Evaluating bert-base-multilingual-cased...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.091769,0.087707,0.985435,0.330888


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.091769,0.087707,0.985435,0.330888
2,0.083659,0.086152,0.985435,0.330888
3,0.096449,0.085042,0.985435,0.330888


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


--- mBERT FINAL RESULTS ---
{'eval_loss': 0.08504164218902588, 'eval_accuracy': 0.9854349400498914, 'eval_f1': 0.3308880152393304, 'eval_runtime': 108.5394, 'eval_samples_per_second': 114.493, 'eval_steps_per_second': 14.317, 'epoch': 3.0}


In [ ]:
# Now run the benchmark for the second model
# This allows you to say "I compared multiple SOTA architectures" in your resume
results_xlmr = train_and_evaluate("xlm-roberta-base")

print("\n--- COMPARATIVE RESEARCH REPORT ---")
print(f"mBERT Accuracy: {results_mbert['eval_accuracy']:.4f} | F1: {results_mbert['eval_f1']:.4f}")
print(f"XLM-R Accuracy: {results_xlmr['eval_accuracy']:.4f} | F1: {results_xlmr['eval_f1']:.4f}")


🚀 Starting Research Phase: Evaluating xlm-roberta-base...


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye


--- COMPARATIVE RESEARCH REPORT ---
mBERT Accuracy: 0.9854 | F1: 0.3309
XLM-R Accuracy: 0.9854 | F1: 0.3309


In [ ]:
import numpy as np
from statsmodels.stats.contingency_tables import mcnemar

def perform_mcnemar(trainer_a, trainer_b, dataset):
    # Get predictions
    preds_a = np.argmax(trainer_a.predict(dataset).predictions, axis=-1)
    preds_b = np.argmax(trainer_b.predict(dataset).predictions, axis=-1)

    # Get ground truth safely
    if hasattr(dataset, "labels"):
        ground_truth = np.array(dataset.labels)
    elif hasattr(dataset, "label_ids"):
        ground_truth = np.array(dataset.label_ids)
    else:
        raise AttributeError("Dataset must contain 'labels' or 'label_ids'")

    # Ensure same length
    min_len = min(len(preds_a), len(preds_b), len(ground_truth))
    preds_a = preds_a[:min_len]
    preds_b = preds_b[:min_len]
    ground_truth = ground_truth[:min_len]

    # Correctness masks
    a_right = preds_a == ground_truth
    b_right = preds_b == ground_truth

    # McNemar contingency table
    table = np.array([
        [np.sum(a_right & b_right), np.sum(a_right & ~b_right)],
        [np.sum(~a_right & b_right), np.sum(~a_right & ~b_right)]
    ])

    print("Contingency Table:\n", table)

    # Run McNemar test
    result = mcnemar(table, exact=True)

    print(f"McNemar's Test Statistic: {result.statistic}")
    print(f"McNemar's Test p-value: {result.pvalue:.6f}")

    if result.pvalue < 0.05:
        print("Result is Statistically Significant")
    else:
        print("Result is NOT Statistically Significant")
